In [1]:
import requests

# Question 1: `search_wikipedia()`




In [37]:

API_URL = "https://en.wikipedia.org/w/api.php"


def search_wikipedia(query: str):

    """
    Search Wikipedia and return search results.

    Parameters
    ----------
    query : str
        Search query.
    limit : int
        Number of results to return.

    Returns
    -------
    list[dict]
        Wikipedia search results.
    """

    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
    
    }

    headers = {
        "User-Agent": "SimpleWikiClient/1.0"
    }

    response = requests.get(API_URL, params=params, headers=headers)
    response.raise_for_status()

    data = response.json()

    return data


In [103]:
results = search_wikipedia('capybara')

len(results['query']['search'])


10

# Question 2. Titles in `search_wikipedia()`

In [73]:
filtered = [r['title'] for r in results['query']['search'] if 'capybara' in (r['title']).lower()]

display(f'Total tiltes with capybara are: {len(filtered)}')
display(filtered)

'Total tiltes with capybara are: 5'

['Capybara',
 'Capybara (disambiguation)',
 'Capybara Games',
 'Lesser capybara',
 'Capybara (software)']

# Question 3. `get_page()`


In [ ]:
def get_page(title: str) -> str:
    """
    Fetch readable (plain text) content of a Wikipedia page.
    """
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "titles": title,
        "explaintext": 1,  
        "exsectionformat": "plain",
    }

    headers = {"User-Agent": "SimpleWikiClient/1.0"}

    response = requests.get(API_URL, params=params, headers=headers)
    response.raise_for_status()

    data = response.json()
    

    # "pages" is a dict keyed by pageid; take the first (and only) page object
    page = next(iter(data["query"]["pages"].values()))

    # If the page doesn't exist, MediaWiki returns {"missing": True}
    if "missing" in page:
        raise ValueError(f'Wikipedia page not found: "{title}"')

    return page.get("extract", "")

In [96]:
capybara_txt = get_page('Capybara')
display(capybara_txt[:500])  # show the first 500 characters
print(len(capybara_txt))


'The capybara or greater capybara (Hydrochoerus hydrochaeris) is the largest living rodent, native to South America. It is a member of the genus Hydrochoerus. Its close relatives include\nguinea pigs and rock cavies, and it is more distantly related to the agouti, the chinchilla, and the nutria. The capybara inhabits savannas and dense forests, and lives near bodies of water. It is a highly social species and can be found in groups as large as one hundred individuals, but usually live in groups of'

13566


# Question 4: Setting up the Agent

In [162]:
from typing import Literal, List, Optional
from pydantic import BaseModel, Field, ValidationError, model_validator
from openai import OpenAI



# =========================================
# Structured response format for the agent's answer
# =========================================



class RAGResponse(BaseModel):
    summary: Optional[str] = Field(default=None, description="Summary of the page")
    intro: Optional[str] = Field(default=None, description="Alias of summary (some models output this)")
    text: Optional[str] = Field(default=None) 

    found_answer: bool = Field(description="True if relevant information was found")

    confidence: float = Field(default=0.8, description="0.0 to 1.0 confidence score")
    confidence_explanation: str = Field(default="Based on retrieved Wikipedia extract.", description="Why this score")

    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(
        default="explanation",
        description="Category of the answer"
    )

    followup_questions: List[str] = Field(default_factory=list, description="Suggested follow-up questions")

    @model_validator(mode="after")
    def normalize_fields(self):
        if not self.summary:
            if self.intro:
                self.summary = self.intro
            elif self.text:
                self.summary = self.text

        if not self.summary:
            raise ValueError("Missing required field: summary (or intro/text)")

        return self

    
# =========================================
# Functions used as tools by the agent  
# =========================================

API_URL = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "SimpleWikiClient/1.0"}

def search_wikipedia(query: str, limit: int = 5) -> dict:
    """
    Search Wikipedia and return total hits + search results.

    Returns
    -------
    dict with keys:
      - totalhits: int
      - results: list[dict]
    """
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
        "srlimit": limit,
    }

    response = requests.get(API_URL, params=params, headers=HEADERS, timeout=15)
    response.raise_for_status()
    data = response.json()

    return {
        "totalhits": data.get("query", {}).get("searchinfo", {}).get("totalhits", 0),
        "results": data.get("query", {}).get("search", []),
    }



def get_page(title: str, intro_only: bool = True) -> dict:
    """
    Fetch readable (plain text) content of a Wikipedia page.

    Returns
    -------
    dict with keys:
      - title: str
      - pageid: int
      - extract: str
    """
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "titles": title,
        "explaintext": 1,
        "exsectionformat": "plain",
    }
    if intro_only:
        params["exintro"] = 1

    response = requests.get(API_URL, params=params, headers=HEADERS, timeout=15)
    response.raise_for_status()
    data = response.json()

    page = next(iter(data["query"]["pages"].values()))
    if "missing" in page:
        raise ValueError(f'Wikipedia page not found: "{title}"')

    return {
        "title": page.get("title", title),
        "pageid": page.get("pageid", 0),
        "extract": page.get("extract", ""),
    }
    
    
# =========================================
# Tools schema 
# =========================================

tool_search_wikipedia = {
    "type": "function",
    "name": "search_wikipedia",
    "description": "Search Wikipedia and return matching pages (titles, pageids, snippets) plus total hits.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query to look up on Wikipedia"},
            "limit": {"type": "integer", "description": "Maximum number of search results to return", "default": 5},
        },
        "required": ["query"],
    },
}

tool_get_page = {
    "type": "function",
    "name": "get_page",
    "description": "Fetch readable plain-text content of a Wikipedia page by title.",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "Wikipedia page title, e.g. 'Capybara'"},
            "intro_only": {"type": "boolean", "description": "If true, return only the intro section", "default": True},
        },
        "required": ["title"],
    },
}




The function calling loop must execute this loop. Just defining the schema is not enough 

- model → “_call search_wikipedia_”

- it runs `search_wikipedia(...)`

- send tool result back

- model → “_call get_page with best title_”

- you run `get_page(...)`

- send tool result back

- model → `final RAGResponse`
 

In [163]:
import json

def make_call(tool_call):
    """
    Execute a single tool call and return a Responses API function_call_output item.
    """
    name = tool_call.name
    arguments = json.loads(tool_call.arguments or "{}")

    try:
        if name == "search_wikipedia":
            result = search_wikipedia(**arguments)
        elif name == "get_page":
            result = get_page(**arguments)
        else:
            result = {"error": f'Unknown tool "{name}"', "arguments": arguments}
    except Exception as e:
        result = {"error": str(e), "tool": name, "arguments": arguments}

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result, ensure_ascii=False),
    }



# =========================================
# Agent loop
# =========================================

openai_client = OpenAI()


RAG_SCHEMA = {
   "summary": "Short factual summary of the page",
    "found_answer": True,
    "confidence": 0.8,
    "confidence_explanation": "Based on retrieved Wikipedia extract.",
    "answer_type": "explanation",
    "followup_questions": ["Example question 1", "Example question 2"]
    
}

def strip_json_fence(text: str) -> str:
    text = (text or "").strip()
    if text.startswith("```"):
        # handle ```json ... ```
        parts = text.split("```")
        if len(parts) >= 3:
            return parts[1].strip()
    return text

# Define the instructions for the agent


instructions = """
You are a Wikipedia-grounded assistant.

You have two tools:
- search_wikipedia(query, limit): search Wikipedia pages
- get_page(title, intro_only): fetch readable plain-text content for a page title

Rules:
- Use tools to gather facts. Do not invent facts.
- If the user provides a Wikipedia URL, infer the page title from it and call get_page.
- If you don't know the best title, call search_wikipedia first, then get_page.
- When you are ready, produce the final answer strictly as JSON matching the RAGResponse schema.
- Your answer is a concise summary of the contents of the page. 
- If you cannot find relevant info, set found_answer=false and explain why.
- Your final answer MUST be valid JSON matching EXACTLY this object structure:

{json.dumps(RAG_SCHEMA, indent=2)}

Additional Rules:
- Return ONLY JSON (no code fences).
- Use ONLY these top-level keys: {list(RAG_SCHEMA.keys())}
- Do NOT output keys like "content", "title".
- Use "summary" for the main text (do not use "intro" or "text").
- answer_type MUST be one of: how-to, explanation, troubleshooting, comparison, reference.

"""



def run_agent(user_query: str, model: str = "gpt-4o-mini", max_iters: int = 8) -> RAGResponse:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_query},
    ]

    tools = [tool_search_wikipedia, tool_get_page]

    print("\n=== AGENT START ===")
    print(f"USER: {user_query}\n")

    for iteration in range(max_iters):
        print(f"\n--- ITERATION {iteration + 1} ---")

        resp = openai_client.responses.create(
            model=model,
            input=messages,
            tools=tools,
        )

        # Detect tool calls
        tool_calls = [item for item in resp.output if item.type == "function_call"]

        if tool_calls:
            print("MODEL decided to call tool(s):")
            messages.extend(resp.output)

            for call in tool_calls:
                args = json.loads(call.arguments or "{}")

                print(f"\n🛠 CALL → {call.name}")
                print(f"   arguments = {args}")

                tool_output = make_call(call)

                # Print short preview of result
                try:
                    preview = json.loads(tool_output["output"])
                    if isinstance(preview, dict) and "extract" in preview:
                        preview_text = (preview["extract"] or "")[:120]
                        print(f"   result preview = {preview_text}...")
                    elif isinstance(preview, dict):
                        print(f"   result keys = {list(preview.keys())}")
                    else:
                        print("   result returned (non-dict)")
                except Exception:
                    print("   result returned (unparseable)")

                messages.append(tool_output)

            continue

        # No tool calls → final answer
        print("MODEL produced final answer (no more tool calls).")

        resp_final = openai_client.responses.create(
            model=model,
            input=messages + resp.output
        )

        raw = resp_final.output_text or ""
        print("\nRaw model output:")
        print(raw[:400], "\n")

        try:
            data = json.loads(strip_json_fence(raw))
            result = RAGResponse.model_validate(data)

            print("✅ Final response validated as RAGResponse")
            print("=== AGENT END ===\n")
            return result

        except (json.JSONDecodeError, ValidationError) as e:
            print("⚠️ Invalid JSON — asking model to fix format")
            print(e)

            messages.append({
                "role": "system",
                "content": (
                    "Reformat your previous response to match EXACTLY this JSON structure. "
                    "Do not add new information. Do not invent facts. Only map fields.\n\n"
                    f"Target JSON template:\n{json.dumps(RAG_SCHEMA, indent=2)}\n\n"
                    f"Allowed keys ONLY: {list(RAG_SCHEMA.keys())}"
                ),
            })

            messages.append({
                "role": "assistant",
                "content": raw
            })

            continue

    raise RuntimeError("Agent did not converge within max_iters.")

# Question 5. Testing Your Agent - Single Page

In [164]:
query = "What is this page about? https://en.wikipedia.org/wiki/Capybara"

result = run_agent(query)
print(result.model_dump_json(indent=2, ensure_ascii=False))


=== AGENT START ===
USER: What is this page about? https://en.wikipedia.org/wiki/Capybara


--- ITERATION 1 ---
MODEL decided to call tool(s):

🛠 CALL → get_page
   arguments = {'title': 'Capybara', 'intro_only': True}
   result preview = The capybara or greater capybara (Hydrochoerus hydrochaeris) is the largest living rodent, native to South America. It i...

--- ITERATION 2 ---
MODEL produced final answer (no more tool calls).

Raw model output:
{
  "found_answer": true,
  "summary": "The capybara (Hydrochoerus hydrochaeris) is the largest living rodent, native to South America. It is part of the genus Hydrochoerus, closely related to guinea pigs and rock cavies. Capybaras inhabit savannas and dense forests, typically near water. They are social animals, often found in groups ranging from 10 to 20, but can gather in groups of up to one hu 

✅ Final response validated as RAGResponse
=== AGENT END ===

{
  "summary": "The capybara (Hydrochoerus hydrochaeris) is the largest living rod

In [166]:
print(result.summary)


The capybara (Hydrochoerus hydrochaeris) is the largest living rodent, native to South America. It is part of the genus Hydrochoerus, closely related to guinea pigs and rock cavies. Capybaras inhabit savannas and dense forests, typically near water. They are social animals, often found in groups ranging from 10 to 20, but can gather in groups of up to one hundred. They are hunted for their meat, hide, and fat.


# Question 6: Testing the Agent

In [167]:
query = "What are the main threats to capybara populations?"

result_threads = run_agent(query)
print(result_threads.model_dump_json(indent=2, ensure_ascii=False))


=== AGENT START ===
USER: What are the main threats to capybara populations?


--- ITERATION 1 ---
MODEL decided to call tool(s):

🛠 CALL → search_wikipedia
   arguments = {'query': 'Capybara threats', 'limit': 5}
   result keys = ['totalhits', 'results']

--- ITERATION 2 ---
MODEL decided to call tool(s):

🛠 CALL → search_wikipedia
   arguments = {'query': 'Capybara', 'limit': 5}
   result keys = ['totalhits', 'results']

--- ITERATION 3 ---
MODEL decided to call tool(s):

🛠 CALL → get_page
   arguments = {'title': 'Capybara', 'intro_only': False}
   result preview = The capybara or greater capybara (Hydrochoerus hydrochaeris) is the largest living rodent, native to South America. It i...

--- ITERATION 4 ---
MODEL produced final answer (no more tool calls).

Raw model output:
{
  "found_answer": true,
  "answer_type": "explanation",
  "summary": "Capybaras (Hydrochoerus hydrochaeris), native to South America, face several threats impacting their populations. They are hunted for thei

# Extra: create an Agent class

In [ ]:

from typing import Any, Callable, Dict, List, Optional, Type

class WikiAgent:
    """
    Wikipedia tool-using agent with a "fake tool" to return structured output.

    Pattern:
      - Model calls real tools (search_wikipedia / get_page)
      - When ready, model calls submit_answer({ ...structured payload... })
      - Runner validates payload with `output_type` and stops
    """

    def __init__(
        self,
        llm_client: OpenAI,
        model: str,
        instructions: str,
        tools: List[dict],
        tool_impls: Dict[str, Callable[..., Any]],
        output_type: Optional[Type[BaseModel]] = None,
        max_iters: int = 8,
        debug: bool = True,
    ):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools
        self.tool_impls = tool_impls
        self.output_type = output_type
        self.max_iters = max_iters
        self.debug = debug

        # Add the fake tool if output_type is provided
        if self.output_type is not None:
            self.tools = self.tools + [self._submit_answer_tool_schema(self.output_type)]

    # ----------------------------
    # Fake tool schema
    # ----------------------------
    def _submit_answer_tool_schema(self, output_type: Type[BaseModel]) -> dict:
        """
        Create a tool schema that asks the model to call submit_answer(payload=...).

        We inject the JSON Schema of the Pydantic model as the tool parameter schema
        so the model is guided to produce the right shape.
        """
        schema = output_type.model_json_schema()

        
        return {
            "type": "function",
            "name": "submit_answer",
            "description": (
                "Call this ONLY when you are ready to return the final structured answer. "
                "The payload must match the required schema."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "payload": schema, 
                },
                "required": ["payload"],
            },
        }

    # ----------------------------
    # Tool dispatcher
    # ----------------------------
    def _execute_tool_call(self, tool_call) -> dict:
        """
        Execute a real tool call and return a function_call_output item.
        """
        name = tool_call.name
        arguments = json.loads(tool_call.arguments or "{}")

        if self.debug:
            print(f"\n🛠 CALL → {name}")
            print(f"   arguments = {arguments}")

        # Stop condition: if the fake tool is called, return the arguments as output (no execution)
        if name == "submit_answer":
            return {
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": json.dumps({"ok": True}),
            }

        fn = self.tool_impls.get(name)
        if fn is None:
            result = {"error": f'Unknown tool "{name}"', "arguments": arguments}
        else:
            try:
                result = fn(**arguments)
            except Exception as e:
                result = {"error": str(e), "tool": name, "arguments": arguments}

        # Preview
        if self.debug:
            if isinstance(result, dict) and "extract" in result:
                prev = (result.get("extract") or "")[:120]
                print(f"   result preview = {prev}...")
            elif isinstance(result, dict):
                print(f"   result keys = {list(result.keys())}")
            else:
                print("   result returned")

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result, ensure_ascii=False),
        }

    # ----------------------------
    # Main loop
    # ----------------------------
    def run(self, user_query: str) -> BaseModel | dict:
        messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": user_query},
        ]

        if self.debug:
            print("\n=== AGENT START ===")
            print(f"USER: {user_query}\n")

        for iteration in range(self.max_iters):
            if self.debug:
                print(f"\n--- ITERATION {iteration + 1} ---")

            resp = self.llm_client.responses.create(
                model=self.model,
                input=messages,
                tools=self.tools,
            )

            
            tool_calls = [item for item in resp.output if item.type == "function_call"]

            if not tool_calls:
               
                if self.debug:
                    print("MODEL did not call any tool. Nudging to call submit_answer when ready.")

                messages.extend(resp.output)
                messages.append({
                    "role": "system",
                    "content": (
                        "When you are ready to respond, call submit_answer with payload matching the schema. "
                        "Do not respond with plain text."
                    ),
                })
                continue

            # Append the model outputs 
            messages.extend(resp.output)

            for call in tool_calls:
                # STOP condition: fake tool called
                if call.name == "submit_answer":
                    args = json.loads(call.arguments or "{}")
                    payload = args.get("payload")

                    if payload is None:
                        # Ask model to try again
                        messages.append({
                            "role": "system",
                            "content": "submit_answer requires a 'payload' object. Call submit_answer again with payload.",
                        })
                        break

                    # Validate payload
                    if self.output_type is not None:
                        try:
                            result = self.output_type.model_validate(payload)
                            if self.debug:
                                print("\n✅ submit_answer received valid payload")
                                print("=== AGENT END ===\n")
                            return result
                        except ValidationError as e:
                            if self.debug:
                                print("\n⚠️ submit_answer payload invalid")
                                print(e)

                            messages.append({
                                "role": "system",
                                "content": (
                                    "Your submit_answer payload did not match the required schema. "
                                    "Fix ONLY the JSON payload and call submit_answer again."
                                ),
                            })
                            # Also acknowledge tool call (optional)
                            messages.append(self._execute_tool_call(call))
                            break

                    # If no output_type, just return raw payload
                    return payload

                # Execute real tools
                tool_output = self._execute_tool_call(call)
                messages.append(tool_output)

        raise RuntimeError("Agent did not converge within max_iters.")
        

    

In [ ]:
# Testing the class

openai_client = OpenAI()

agent = WikiAgent(
    llm_client=openai_client,
    model="gpt-4o-mini",
    instructions=instructions,
    tools=[tool_search_wikipedia, tool_get_page],
    tool_impls={
        "search_wikipedia": search_wikipedia,
        "get_page": get_page,
    },
    output_type=RAGResponse,   # <— enables fake tool submit_answer
    max_iters=10,
    debug=True,
)

result = agent.run("What are the main threats to capybara populations?")
print(result.model_dump_json(indent=2, ensure_ascii=False))


=== AGENT START ===
USER: What are the main threats to capybara populations?


--- ITERATION 1 ---

🛠 CALL → search_wikipedia
   arguments = {'query': 'Capybara threats', 'limit': 5}
   result keys = ['totalhits', 'results']

--- ITERATION 2 ---

🛠 CALL → search_wikipedia
   arguments = {'query': 'capybara', 'limit': 5}
   result keys = ['totalhits', 'results']

--- ITERATION 3 ---

🛠 CALL → get_page
   arguments = {'title': 'Capybara', 'intro_only': False}
   result preview = The capybara or greater capybara (Hydrochoerus hydrochaeris) is the largest living rodent, native to South America. It i...

--- ITERATION 4 ---

✅ submit_answer received valid payload
=== AGENT END ===

{
  "summary": "Capybara populations face several threats, primarily from hunting and habitat destruction. They are hunted for their meat and skin, and in some regions, they are viewed as competitors for livestock grazing. Although they are not considered a threatened species overall, localized hunting pressures